<a href="https://colab.research.google.com/github/athishsreeram/tubenotebook/blob/main/top3videos_metadata_transcript_preview_and_saves_everything_to_CSV_and_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ─────────────────────────────
# Install dependencies (run once in Colab)
# ─────────────────────────────
!pip install yt-dlp youtube-transcript-api --upgrade

# ─────────────────────────────
# Imports
# ─────────────────────────────
import yt_dlp
import json
import csv
from datetime import datetime
from youtube_transcript_api import YouTubeTranscriptApi

# ─────────────────────────────
# Configuration
# ─────────────────────────────
CHANNEL_URL = "https://www.youtube.com/@rajshamani/videos"  # ← Change this
MAX_VIDEOS = 50        # How many videos to fetch initially
TOP_N = 3              # Only analyze top N videos
TRANSCRIPT_LANGS = ["en"]  # Any language list, e.g., ["en","hi"]

# ─────────────────────────────
# Fetch channel videos
# ─────────────────────────────
def get_channel_videos(channel_url, max_videos=None):
    ydl_opts = {
        "quiet": True,
        "extract_flat": True,
        "skip_download": True,
        "ignoreerrors": True,
        "playlistend": max_videos,
    }

    videos = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)
        entries = info.get("entries", [])
        for entry in entries:
            if not entry:
                continue
            video_id = entry.get("id")
            url = f"https://www.youtube.com/watch?v={video_id}"
            videos.append({
                "title": entry.get("title"),
                "url": url,
                "video_id": video_id,
                "view_count": entry.get("view_count") or 0,
                "duration": entry.get("duration"),
                "upload_date": entry.get("upload_date"),
            })
    return videos

# ─────────────────────────────
# Fetch transcript
# ─────────────────────────────
def get_transcript(video_id, langs=None):
    if langs is None:
        langs = ["en"]
    try:
        transcript = YouTubeTranscriptApi().fetch(video_id, languages=langs)
        full_text = " ".join([t.text for t in transcript])
        segments = [{"text": t.text, "start": t.start, "duration": t.duration} for t in transcript]
        return {"video_id": video_id, "transcript": full_text, "segments": segments}
    except Exception as e:
        print(f"⚠️ Transcript unavailable for {video_id}: {e}")
        return None

# ─────────────────────────────
# Enrich video metadata
# ─────────────────────────────
def enrich_video(video):
    try:
        ydl_opts = {"quiet": True, "skip_download": True}
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video["url"], download=False)
            video["description"] = info.get("description")
            video["like_count"] = info.get("like_count")
            video["comment_count"] = info.get("comment_count")
            video["tags"] = info.get("tags")
            video["categories"] = info.get("categories")
            video["thumbnail"] = info.get("thumbnail")
            video["uploader"] = info.get("uploader")
            video["channel_id"] = info.get("channel_id")
            # Engagement score
            views = video.get("view_count") or 1
            likes = video.get("like_count") or 0
            comments = video.get("comment_count") or 0
            video["engagement_score"] = round((likes + comments) / views, 5)
            # Transcript
            transcript_data = get_transcript(video["video_id"], TRANSCRIPT_LANGS)
            if transcript_data:
                video["transcript"] = transcript_data["transcript"]
                video["transcript_segments"] = transcript_data["segments"]
            else:
                video["transcript"] = None
                video["transcript_segments"] = None
    except Exception as e:
        print(f"❌ Failed to enrich video {video['video_id']}: {e}")
    return video

# ─────────────────────────────
# Save results
# ─────────────────────────────
def save_to_json(videos):
    filename = f"youtube_top_videos_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(videos, f, indent=2, ensure_ascii=False)
    print(f"💾 Saved JSON: {filename}")

def save_to_csv(videos):
    filename = f"youtube_top_videos_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    keys = set()
    for v in videos:
        keys.update(v.keys())
    keys = list(keys)
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(videos)
    print(f"💾 Saved CSV: {filename}")

# ─────────────────────────────
# Run pipeline
# ─────────────────────────────
print("🚀 Fetching videos...")
videos = get_channel_videos(CHANNEL_URL, MAX_VIDEOS)
print(f"✅ Found {len(videos)} videos")

# Sort by view count and take top N
videos = sorted(videos, key=lambda x: x["view_count"] or 0, reverse=True)[:TOP_N]

print(f"\n🔥 Enriching top {TOP_N} videos...")
for i, v in enumerate(videos, 1):
    print(f"🔍 [{i}] {v['title'][:50]}")
    enrich_video(v)

# Print results
print("\n📋 Top videos info:")
for i, v in enumerate(videos, 1):
    print(f"\n[{i}] {v['title']}")
    print(f"URL: {v['url']}")
    print(f"Views: {v['view_count']}, Likes: {v.get('like_count')}, Comments: {v.get('comment_count')}")
    print(f"Transcript preview: {v['transcript'][:200] if v['transcript'] else 'No transcript'}")

# Save results
save_to_json(videos)
save_to_csv(videos)

print("\n🎯 DONE — Top 3 videos enriched with transcripts!")

🚀 Fetching videos...
✅ Found 50 videos

🔥 Enriching top 3 videos...
🔍 [1] President of France on Trump, India, Modi, Tech & 


🔍 [2] Sunita Williams On 286 Days in Space, NASA Mission


🔍 [3] Vikas Divyakirti on India, China, Power Struggles,


⚠️ Transcript unavailable for 2hl-uVjXZzc: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=2hl-uVjXZzc! This is most likely caused by:

No transcripts were found for any of the requested language codes: ['en']

For this video (2hl-uVjXZzc) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - hi ("Hindi (auto-generated)")

(TRANSLATION LANGUAGES)
None

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!

📋 Top videos info:

[1] President of France on Trump, India, Modi, Tech & Future | H.E. Emmanuel Macron | FO473 Raj Shamani
URL: https://www.youtube.com/watch?v=9